# Gan 2026 Living Observatory

Purpose: keep one runnable notebook for loading checks, gold-label distribution, scoring, and failure slices while candidate pipelines evolve.

This is a development-control surface, not a benchmark claim. Use it to inspect train/validation behavior, candidate artifacts, and row families; keep the locked test split out of routine analysis.

Core docs to keep in view:

- `PROJECT_STATUS.md` for the current objective, work board, and claim caveats.
- `docs/research/contribution_thesis.md` for the modular, generalizable, transparent hybrid thesis.
- `docs/design/data_contract.md` for row validity, source labels, and benchmark data handling.
- `docs/design/gan2026_split_protocol.md` for split discipline and validation-escalation rules.
- `docs/research/gan2026_architecture_space_2026-06-01.md` for the architecture gate before the metric gate.

## Setup

Run from either the repo root or the `notebooks/` directory. The cell finds the repo root and uses the package APIs rather than reimplementing loading or scoring logic.

In [ ]:
from __future__ import annotations

import json
import sys
from pathlib import Path

import pandas as pd


def find_repo_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for candidate in (start, *start.parents):
        if (candidate / "pyproject.toml").exists() and (candidate / "src").exists():
            return candidate
    raise RuntimeError("Could not find repo root from current working directory")


REPO_ROOT = find_repo_root()
if str(REPO_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(REPO_ROOT / "src"))

from clinical_extraction.tasks.seizure_frequency.gan2026.data import (  # noqa: E402
    DEFAULT_DATA_PATH,
    DEFAULT_SPLIT_MANIFEST_PATH,
    load_records_for_split,
    load_split_manifest,
)
from clinical_extraction.tasks.seizure_frequency.gan2026.evaluate import (  # noqa: E402
    convert_to_categories,
    evaluate_predictions,
)

pd.set_option("display.max_columns", 80)
pd.set_option("display.max_colwidth", 140)

DATA_PATH = REPO_ROOT / DEFAULT_DATA_PATH
SPLIT_MANIFEST_PATH = REPO_ROOT / DEFAULT_SPLIT_MANIFEST_PATH
manifest = load_split_manifest(SPLIT_MANIFEST_PATH)

REPO_ROOT

## Loading And Split Shape

This pins the basic data contract: source row index, split membership, `row_ok`, gold label semantics, and scorer categories.

In [10]:
def records_to_frame(records, split: str | None = None) -> pd.DataFrame:
    rows = []
    for record in records:
        rows.append(
            {
                "split": split,
                "source_row_index": record.source_row_index,
                "gold_label": record.gold_label,
                "gold_normalized_label": record.gold_normalized_label,
                "gold_label_kind": str(record.gold_label_kind),
                "gold_monthly_frequency": record.gold_monthly_frequency,
                "gold_purist_category": convert_to_categories(
                    [record.gold_monthly_frequency], method="purist"
                )[0],
                "gold_pragmatic_category": convert_to_categories(
                    [record.gold_monthly_frequency], method="pragmatic"
                )[0],
                "note_chars": len(record.note_text),
                "gold_reference_chars": len(record.gold_reference),
            }
        )
    return pd.DataFrame(rows)


split_frames = {
    split: records_to_frame(
        load_records_for_split(split, data_path=DATA_PATH, manifest_path=SPLIT_MANIFEST_PATH),
        split=split,
    )
    for split in manifest["splits"]
}
all_records_df = pd.concat(split_frames.values(), ignore_index=True)

split_summary = (
    all_records_df.groupby("split", observed=True)
    .agg(
        rows=("source_row_index", "count"),
        mean_note_chars=("note_chars", "mean"),
    )
    .assign(mean_note_chars=lambda frame: frame["mean_note_chars"].round(1))
)
split_summary

,rows,mean_note_chars
split,,
test,450,2752.6
train,300,2774.0
validation,750,2735.5


## Gold-Label Distribution

These tables show the target surface before candidate behavior enters the picture. Use them when interpreting score movement: a candidate can look good overall while failing a clinically important sparse slice.

In [3]:
def distribution_table(df: pd.DataFrame, column: str) -> pd.DataFrame:
    counts = df.groupby(["split", column], observed=True).size().rename("n").reset_index()
    totals = df.groupby("split", observed=True).size().rename("split_n")
    return (
        counts.join(totals, on="split")
        .assign(pct=lambda frame: (100 * frame["n"] / frame["split_n"]).round(1))
        .sort_values(["split", "n", column], ascending=[True, False, True])
        .reset_index(drop=True)
    )


gold_kind_distribution = distribution_table(all_records_df, "gold_label_kind")
purist_distribution = distribution_table(all_records_df, "gold_purist_category")
pragmatic_distribution = distribution_table(all_records_df, "gold_pragmatic_category")

display(gold_kind_distribution)
display(purist_distribution)
display(pragmatic_distribution)

,split,gold_label_kind,n,split_n,pct
0,test,frequency,281,450,62.4
1,test,seizure_free,67,450,14.9
2,test,unknown,60,450,13.3
3,test,unresolved_multiple,26,450,5.8
4,test,no_reference,16,450,3.6
5,train,frequency,188,300,62.7
6,train,seizure_free,44,300,14.7
7,train,unknown,40,300,13.3
8,train,unresolved_multiple,17,300,5.7
9,train,no_reference,11,300,3.7


,split,gold_purist_category,n,split_n,pct
0,test,seizure_freq_unknown,102,450,22.7
1,test,seizure_freq_more1week_less1day,98,450,21.8
2,test,currently_no_seizure,67,450,14.9
3,test,seizure_freq_more1mon_less1week,62,450,13.8
4,test,seizure_freq_more1per6mon_less1mon,52,450,11.6
5,test,seizure_freq_1ormore_daily,36,450,8.0
6,test,seizure_freq_1_per_mon,20,450,4.4
7,test,seizure_freq_1_per_week,6,450,1.3
8,test,seizure_freq_1_per_yr,6,450,1.3
9,test,seizure_freq_1_per_6mon,1,450,0.2


,split,gold_pragmatic_category,n,split_n,pct
0,test,seizure_frequent,202,450,44.9
1,test,seizure_freq_unknown,102,450,22.7
2,test,seizure_infrequent,79,450,17.6
3,test,currently_no_seizure,67,450,14.9
4,train,seizure_frequent,141,300,47.0
5,train,seizure_freq_unknown,68,300,22.7
6,train,seizure_infrequent,47,300,15.7
7,train,currently_no_seizure,44,300,14.7
8,validation,seizure_frequent,346,750,46.1
9,validation,seizure_freq_unknown,170,750,22.7


## Scoring Helpers

Use repo scoring functions for Purist and Pragmatic results. The identity check below is only a smoke test that gold labels round-trip through the scorer.

In [4]:
def metrics_frame(y_true, y_pred) -> pd.DataFrame:
    rows = []
    for method in ("purist", "pragmatic"):
        metrics = evaluate_predictions(y_true, y_pred, method=method)
        for averaging, values in metrics.items():
            rows.append({"method": method, "averaging": averaging, **values})
    return pd.DataFrame(rows)


def score_prediction_frame(
    frame: pd.DataFrame,
    prediction_column: str = "prediction_monthly_frequency",
    gold_column: str = "gold_monthly_frequency",
) -> pd.DataFrame:
    scored = frame.dropna(subset=[gold_column, prediction_column]).copy()
    return metrics_frame(scored[gold_column].astype(float), scored[prediction_column].astype(float))


identity_rows = all_records_df.assign(
    prediction_monthly_frequency=lambda frame: frame["gold_monthly_frequency"]
)
score_prediction_frame(identity_rows)

,method,averaging,precision,recall,f1,accuracy
0,purist,micro,1.0,1.0,1.0,1.0
1,purist,macro,1.0,1.0,1.0,1.0
2,purist,weighted,1.0,1.0,1.0,1.0
3,pragmatic,micro,1.0,1.0,1.0,1.0
4,pragmatic,macro,1.0,1.0,1.0,1.0
5,pragmatic,weighted,1.0,1.0,1.0,1.0


## Run Registry And Ladder

The run registry is the canonical spine for the living notebook. It keeps model role, split, replay status, repair mode, decision, and claim-language caveats attached to metrics so we do not accidentally treat a replay, saturated prefix, or analysis-only artifact as a clean benchmark result.

In [ ]:
REGISTRY_PATH = REPO_ROOT / "experiments/registry.jsonl"


def load_jsonl(path: Path) -> list[dict]:
    if not path.exists():
        return []
    with path.open(encoding="utf-8") as handle:
        return [json.loads(line) for line in handle if line.strip()]


def load_registry_frame(path: Path = REGISTRY_PATH) -> pd.DataFrame:
    rows = load_jsonl(path)
    if not rows:
        return pd.DataFrame()
    frame = pd.DataFrame(rows)
    frame["primary_metrics"] = frame["primary_metrics"].apply(lambda value: value or {})
    frame["artifact_paths"] = frame["artifact_paths"].apply(tuple)
    frame["date"] = pd.to_datetime(frame["date"], errors="coerce")
    return frame.sort_values(["date", "run_id"], ascending=[False, True]).reset_index(drop=True)


def metric_from_map(metrics: dict, candidates: tuple[str, ...]) -> object:
    for key in candidates:
        if key in metrics:
            return metrics[key]
    return None


def run_surface_stage(row: pd.Series) -> str:
    split = str(row.get("split", ""))
    row_count = int(row.get("row_count") or 0)
    if split == "validation":
        if row_count <= 25:
            return "25 smoke"
        if row_count <= 50:
            return "50 signal"
        if row_count <= 250:
            return "250 decision gate"
        return "750 rare full validation"
    if split == "test":
        return "locked test"
    if "test" in split:
        return "mixed validation/test context"
    if "hard" in split or "synthetic" in split:
        return "hard-case mechanism probe"
    return split or "unknown"


def saturation_or_claim_guardrail(row: pd.Series) -> str:
    split = str(row.get("split", ""))
    decision = str(row.get("decision", ""))
    metrics = row.get("primary_metrics") or {}
    row_count = int(row.get("row_count") or 0)
    purist_correct = metric_from_map(
        metrics,
        (
            "deterministic_purist_correct",
            "deterministic_top_purist_correct",
            "raw_purist_correct",
            "gated_purist_correct",
            "clean_purist_correct",
            "purist_correct",
            "adjudicator_purist_correct",
        ),
    )
    try:
        purist_rate = float(purist_correct) / row_count if purist_correct is not None and row_count else None
    except (TypeError, ValueError):
        purist_rate = None
    if "test" in split:
        return "do not inspect row-level test failures during development"
    if split == "validation" and row_count in {25, 50}:
        return "contract/signal prefix; do not promote on aggregate F1"
    if split == "validation" and purist_rate is not None and purist_rate >= 0.95:
        return "near-saturated validation surface; prefer hard slices or selective-action analysis"
    if "hard" in split or "synthetic" in split:
        return "mechanism probe, not benchmark evidence"
    if decision in {"reject", "revise"}:
        return "read claim-language notes before reuse"
    return "ordinary development context"


registry_df = load_registry_frame()
if registry_df.empty:
    print(f"No registry rows found at {REGISTRY_PATH}")
else:
    registry_ladder = registry_df.assign(
        surface_stage=lambda frame: frame.apply(run_surface_stage, axis=1),
        guardrail=lambda frame: frame.apply(saturation_or_claim_guardrail, axis=1),
        purist_signal=lambda frame: frame["primary_metrics"].apply(
            lambda metrics: metric_from_map(
                metrics,
                (
                    "validation_purist",
                    "test_purist",
                    "clean_purist_correct",
                    "purist_correct",
                    "adjudicator_purist_correct",
                    "gated_purist_correct",
                    "raw_purist_correct",
                    "deterministic_purist_correct",
                ),
            )
        ),
        pragmatic_signal=lambda frame: frame["primary_metrics"].apply(
            lambda metrics: metric_from_map(
                metrics,
                (
                    "validation_pragmatic",
                    "test_pragmatic",
                    "clean_pragmatic_correct",
                    "pragmatic_correct",
                    "adjudicator_pragmatic_correct",
                    "gated_pragmatic_correct",
                    "deterministic_pragmatic_correct",
                ),
            )
        ),
    )[
        [
            "run_id",
            "date",
            "pipeline_family",
            "split",
            "row_count",
            "surface_stage",
            "decision",
            "replay_status",
            "model_role",
            "repair_mode",
            "purist_signal",
            "pragmatic_signal",
            "guardrail",
            "claim_language_notes",
        ]
    ]
    display(registry_ladder)

## Registry-Driven Artifact Selection

Choose one or two `run_id` values from the ladder. JSONL artifacts become row-level prediction frames; JSON analysis artifacts become compact mechanism summaries. Keep locked-test artifacts for aggregate context only unless the candidate was frozen and the review is explicitly post-hoc.

In [ ]:
DEFAULT_COMPARE_RUN_IDS = [
    "gan2026_hybrid_adjudicator_v01_validation250_schema_replay_2026-06-01",
    "gan2026_hybrid_adjudicator_v02_validation250_live_2026-06-01",
    "gan2026_hybrid_adjudicator_v02_cluster_diary_candidate_recall_synthetic_hard_case_component_stress_2026-06-01",
]

COMPARE_RUN_IDS = [run_id for run_id in DEFAULT_COMPARE_RUN_IDS if not registry_df.empty and run_id in set(registry_df["run_id"])]
if not COMPARE_RUN_IDS and not registry_df.empty:
    COMPARE_RUN_IDS = registry_df["run_id"].head(2).tolist()


def registry_record(run_id: str) -> dict:
    matches = registry_df[registry_df["run_id"] == run_id]
    if matches.empty:
        raise KeyError(f"Unknown run_id: {run_id}")
    return matches.iloc[0].to_dict()


def artifact_paths_for_run(run_id: str, suffixes: tuple[str, ...] | None = None) -> list[Path]:
    record = registry_record(run_id)
    paths = [REPO_ROOT / path for path in record.get("artifact_paths", ())]
    if suffixes is not None:
        paths = [path for path in paths if path.suffix in suffixes]
    return paths


def first_jsonl_artifact(run_id: str) -> Path | None:
    paths = artifact_paths_for_run(run_id, suffixes=(".jsonl",))
    return paths[0] if paths else None


def json_analysis_artifacts(run_id: str) -> list[Path]:
    return artifact_paths_for_run(run_id, suffixes=(".json",))


selected_runs = pd.DataFrame([registry_record(run_id) for run_id in COMPARE_RUN_IDS]) if COMPARE_RUN_IDS else pd.DataFrame()
if selected_runs.empty:
    print("No comparable registry runs available.")
else:
    display(selected_runs[["run_id", "split", "row_count", "decision", "replay_status", "repair_mode", "claim_language_notes"]])
    for run_id in COMPARE_RUN_IDS:
        print(f"\n{run_id}")
        print("  JSONL:", first_jsonl_artifact(run_id))
        print("  JSON summaries:", [str(path.relative_to(REPO_ROOT)) for path in json_analysis_artifacts(run_id)])

## Candidate Artifact Loader

By default this loads the first JSONL artifact for `SELECTED_RUN_ID`. The flattener handles common structured LLM `score_layers`, hybrid adjudicator `scores`, and older direct `comparison` records. It keeps parse/schema failures separate from scorable rows and preserves enough row metadata for transition analysis.

In [ ]:
SELECTED_RUN_ID = COMPARE_RUN_IDS[0] if COMPARE_RUN_IDS else None
ARTIFACT_PATH = first_jsonl_artifact(SELECTED_RUN_ID) if SELECTED_RUN_ID else REPO_ROOT / "experiments/gan2026_clean_attribution_format50_v0_2026-06-01.jsonl"


def load_jsonl(path: Path) -> list[dict]:
    if not path.exists():
        return []
    with path.open(encoding="utf-8") as handle:
        return [json.loads(line) for line in handle if line.strip()]


def preferred_score_layer(row: dict) -> tuple[str | None, dict]:
    score_layers = row.get("score_layers") or {}
    scores = row.get("scores") or {}
    comparison = row.get("comparison") or {}
    for name in (
        "clean_scorer_facing",
        "conservative_adjudicator",
        "adjudicator",
        "raw_adjudicator",
        "strict_format",
        "raw",
        "deterministic_top",
    ):
        if name in score_layers:
            return name, score_layers[name]
        if name in scores:
            return name, scores[name]
    if comparison:
        return "comparison", comparison
    return None, {}


def final_label_from_row(row: dict, layer: dict) -> object:
    structured = row.get("structured_record") or {}
    selection = structured.get("selection") or structured.get("final_query") or {}
    decision_record = row.get("decision_record") or {}
    return (
        layer.get("final_label")
        or layer.get("prediction_label")
        or selection.get("final_label")
        or decision_record.get("final_label")
    )


def event_summary(row: dict) -> tuple[int, str, int]:
    normalized_events = row.get("normalized_events") or []
    if not normalized_events:
        deterministic = row.get("deterministic_diagnostics") or {}
        normalized_events = deterministic.get("normalized_events") or deterministic.get("candidate_events") or []
    event_kinds = [event.get("semantic_kind") or event.get("kind") for event in normalized_events if isinstance(event, dict)]
    return (
        len(normalized_events),
        ", ".join(str(kind) for kind in event_kinds if kind),
        sum(str(kind) == "cluster" for kind in event_kinds if kind),
    )


def artifact_records_to_frame(rows: list[dict], run_id: str | None = None) -> pd.DataFrame:
    flattened = []
    for row in rows:
        layer_name, layer = preferred_score_layer(row)
        reference = row.get("reference") or {}
        structured = row.get("structured_record") or {}
        selection = structured.get("selection") or structured.get("final_query") or {}
        decision_record = row.get("decision_record") or {}
        parse_errors = row.get("parse_errors") or []
        event_count, event_kinds, cluster_event_count = event_summary(row)
        candidate_recall = row.get("candidate_recall") or {}
        conservative_gate = row.get("conservative_gate") or {}
        predicted_monthly = layer.get("predicted_monthly_frequency")
        flattened.append(
            {
                "run_id": run_id,
                "source_row_index": row.get("source_row_index"),
                "split": row.get("split"),
                "prompt_version": row.get("prompt_version"),
                "score_layer": layer_name,
                "gold_label": reference.get("gold_label"),
                "gold_label_kind": reference.get("gold_label_kind"),
                "gold_monthly_frequency": layer.get(
                    "gold_monthly_frequency", reference.get("gold_monthly_frequency")
                ),
                "prediction_label": final_label_from_row(row, layer),
                "prediction_kind": selection.get("final_kind") or selection.get("answer_kind"),
                "prediction_monthly_frequency": predicted_monthly,
                "purist_correct": layer.get("purist_correct"),
                "pragmatic_correct": layer.get("pragmatic_correct"),
                "gold_purist_category": layer.get("gold_purist_category"),
                "predicted_purist_category": layer.get("predicted_purist_category"),
                "gold_pragmatic_category": layer.get("gold_pragmatic_category"),
                "predicted_pragmatic_category": layer.get("predicted_pragmatic_category"),
                "scorable": layer.get("scorable"),
                "evidence_valid": row.get("evidence_valid") or (row.get("evidence_summary") or {}).get("selected_evidence_exact"),
                "parse_error_count": len(parse_errors),
                "parse_errors": "; ".join(str(error) for error in parse_errors),
                "event_count": event_count,
                "event_kinds": event_kinds,
                "cluster_event_count": cluster_event_count,
                "candidate_count": candidate_recall.get("candidate_count"),
                "candidate_purist_recalled": candidate_recall.get("purist_category_recalled"),
                "used_deterministic_fallback": conservative_gate.get("used_deterministic_fallback"),
                "gate_reasons": ", ".join(conservative_gate.get("fired_gates") or []),
                "selected_evidence": selection.get("evidence") or decision_record.get("rationale"),
                "rationale": selection.get("rationale") or decision_record.get("rationale"),
            }
        )
    return pd.DataFrame(flattened)


artifact_rows = load_jsonl(ARTIFACT_PATH)
artifact_df = artifact_records_to_frame(artifact_rows, run_id=SELECTED_RUN_ID)
print(f"Loaded {len(artifact_df)} rows from {ARTIFACT_PATH}")
artifact_df.head()

In [ ]:
artifact_columns = [
    "run_id",
    "source_row_index",
    "score_layer",
    "gold_label",
    "prediction_label",
    "purist_correct",
    "pragmatic_correct",
    "parse_error_count",
    "event_count",
    "candidate_count",
    "candidate_purist_recalled",
    "used_deterministic_fallback",
    "gate_reasons",
    "selected_evidence",
    "rationale",
]
available_artifact_columns = [column for column in artifact_columns if column in artifact_df.columns]
artifact_df[available_artifact_columns]

In [ ]:
if artifact_df.empty:
    print(f"No artifact rows found at {ARTIFACT_PATH}")
else:
    artifact_summary = pd.DataFrame(
        [
            {
                "run_id": SELECTED_RUN_ID,
                "score_layer": artifact_df["score_layer"].dropna().mode().iat[0] if artifact_df["score_layer"].notna().any() else None,
                "rows": len(artifact_df),
                "scorable_rows": int(artifact_df["prediction_monthly_frequency"].notna().sum()),
                "purist_correct_rows": int(artifact_df["purist_correct"].eq(True).sum()),
                "pragmatic_correct_rows": int(artifact_df["pragmatic_correct"].eq(True).sum()),
                "purist_all_row_accuracy": round(artifact_df["purist_correct"].eq(True).mean(), 4),
                "pragmatic_all_row_accuracy": round(artifact_df["pragmatic_correct"].eq(True).mean(), 4),
                "parse_error_rows": int((artifact_df["parse_error_count"] > 0).sum()),
                "deterministic_fallback_rows": int(artifact_df.get("used_deterministic_fallback", pd.Series(dtype=object)).eq(True).sum()),
                "candidate_recall_rows": int(artifact_df.get("candidate_purist_recalled", pd.Series(dtype=object)).eq(True).sum()),
            }
        ]
    )
    display(artifact_summary)
    display(score_prediction_frame(artifact_df))

## Attribution And Repair Layers

When artifacts expose multiple scoring layers, inspect them as a ladder instead of collapsing to one final answer. This helps distinguish raw model behavior, schema/format repair, clean scorer-facing policy, deterministic fallback, and gated hybrid behavior.

In [ ]:
def artifact_layers_to_frame(rows: list[dict], run_id: str | None = None) -> pd.DataFrame:
    layer_rows = []
    for row in rows:
        reference = row.get("reference") or {}
        layer_maps = []
        if row.get("score_layers"):
            layer_maps.extend((name, payload) for name, payload in row["score_layers"].items())
        if row.get("scores"):
            layer_maps.extend((name, payload) for name, payload in row["scores"].items())
        if not layer_maps and row.get("comparison"):
            layer_maps.append(("comparison", row["comparison"]))
        for layer_name, layer in layer_maps:
            layer_rows.append(
                {
                    "run_id": run_id,
                    "source_row_index": row.get("source_row_index"),
                    "split": row.get("split"),
                    "layer": layer_name,
                    "gold_label": reference.get("gold_label"),
                    "final_label": layer.get("final_label") or layer.get("prediction_label"),
                    "gold_monthly_frequency": layer.get("gold_monthly_frequency", reference.get("gold_monthly_frequency")),
                    "predicted_monthly_frequency": layer.get("predicted_monthly_frequency"),
                    "purist_correct": layer.get("purist_correct"),
                    "pragmatic_correct": layer.get("pragmatic_correct"),
                    "predicted_purist_category": layer.get("predicted_purist_category"),
                    "gold_purist_category": layer.get("gold_purist_category"),
                    "scorable": layer.get("scorable"),
                }
            )
    return pd.DataFrame(layer_rows)


layer_df = artifact_layers_to_frame(artifact_rows, run_id=SELECTED_RUN_ID)
if layer_df.empty:
    print("Selected artifact has no explicit score_layers/scores ladder.")
else:
    layer_summary = (
        layer_df.groupby("layer", observed=True)
        .agg(
            rows=("source_row_index", "count"),
            scorable_rows=("predicted_monthly_frequency", lambda value: int(value.notna().sum())),
            purist_correct=("purist_correct", lambda value: int(value.eq(True).sum())),
            pragmatic_correct=("pragmatic_correct", lambda value: int(value.eq(True).sum())),
        )
        .assign(
            purist_rate=lambda frame: (frame["purist_correct"] / frame["rows"]).round(4),
            pragmatic_rate=lambda frame: (frame["pragmatic_correct"] / frame["rows"]).round(4),
        )
        .reset_index()
    )
    display(layer_summary)

    layer_pivot = layer_df.pivot_table(
        index="source_row_index",
        columns="layer",
        values="purist_correct",
        aggfunc="first",
    )
    display(layer_pivot.head(20))

## Cross-Run Row Transitions

This aligns two JSONL runs by `source_row_index`. The highest-signal cells are wrong-to-correct, correct-to-wrong, and changed predictions among rows both systems score. Use this before trusting small aggregate gains.

In [ ]:
def load_run_frame(run_id: str) -> pd.DataFrame:
    artifact_path = first_jsonl_artifact(run_id)
    if artifact_path is None:
        return pd.DataFrame()
    return artifact_records_to_frame(load_jsonl(artifact_path), run_id=run_id)


run_frames = {run_id: load_run_frame(run_id) for run_id in COMPARE_RUN_IDS}
row_level_run_ids = [run_id for run_id, frame in run_frames.items() if not frame.empty]

if len(row_level_run_ids) < 2:
    print("Need two JSONL-backed runs for row-level transition analysis.")
else:
    RUN_A, RUN_B = row_level_run_ids[:2]
    a = run_frames[RUN_A].add_prefix("a_")
    b = run_frames[RUN_B].add_prefix("b_")
    transitions = a.merge(
        b,
        left_on="a_source_row_index",
        right_on="b_source_row_index",
        how="inner",
    )
    transitions["transition"] = transitions.apply(
        lambda row: (
            "A wrong -> B correct"
            if row["a_purist_correct"] is not True and row["b_purist_correct"] is True
            else "A correct -> B wrong"
            if row["a_purist_correct"] is True and row["b_purist_correct"] is not True
            else "both correct"
            if row["a_purist_correct"] is True and row["b_purist_correct"] is True
            else "both wrong, changed label"
            if row["a_prediction_label"] != row["b_prediction_label"]
            else "both wrong, same label"
        ),
        axis=1,
    )
    transition_summary = (
        transitions.groupby("transition", observed=True)
        .size()
        .rename("rows")
        .reset_index()
        .sort_values("rows", ascending=False)
    )
    print(f"A = {RUN_A}")
    print(f"B = {RUN_B}")
    display(transition_summary)
    display(
        transitions[
            [
                "a_source_row_index",
                "transition",
                "a_gold_label",
                "a_prediction_label",
                "b_prediction_label",
                "a_gold_purist_category",
                "a_predicted_purist_category",
                "b_predicted_purist_category",
                "a_parse_errors",
                "b_parse_errors",
                "a_selected_evidence",
                "b_selected_evidence",
            ]
        ]
        .sort_values(["transition", "a_source_row_index"])
        .head(80)
    )

## Hard-Slice And Selective-Action Summaries

Registry runs often point to JSON analysis artifacts rather than row-level predictions. These summaries keep validation hard slices, synthetic hard-case families, and selective-action reports in the same notebook so saturated-surface decisions are based on mechanism evidence.

In [ ]:
def load_json(path: Path) -> object:
    return json.loads(path.read_text(encoding="utf-8"))


def summarize_analysis_artifact(path: Path) -> pd.DataFrame:
    data = load_json(path)
    rows = []
    kind = data.get("artifact_kind", path.stem) if isinstance(data, dict) else path.stem
    if isinstance(data, dict) and isinstance(data.get("selective_actions"), dict):
        for action, payload in data["selective_actions"].items():
            row = {"artifact": path.name, "kind": kind, "section": "selective_actions", "name": action}
            if isinstance(payload, dict):
                row.update({key: value for key, value in payload.items() if not isinstance(value, (dict, list))})
            rows.append(row)
    if isinstance(data, dict) and isinstance(data.get("slice_selective_actions"), dict):
        for slice_name, payload in data["slice_selective_actions"].items():
            row = {"artifact": path.name, "kind": kind, "section": "slice_selective_actions", "name": slice_name}
            if isinstance(payload, dict):
                row.update({key: value for key, value in payload.items() if not isinstance(value, (dict, list))})
            rows.append(row)
    hard_slices = data.get("validation_hard_slices", data) if isinstance(data, dict) else {}
    if isinstance(hard_slices, dict) and isinstance(hard_slices.get("slices"), list):
        for payload in hard_slices["slices"]:
            row = {"artifact": path.name, "kind": kind, "section": "validation_hard_slices", "name": payload.get("slice_name")}
            row.update({key: value for key, value in payload.items() if key != "members" and not isinstance(value, (dict, list))})
            rows.append(row)
    if isinstance(data, dict) and isinstance(data.get("family_summaries"), dict):
        for family, payload in data["family_summaries"].items():
            row = {"artifact": path.name, "kind": kind, "section": "family_summaries", "name": family}
            if isinstance(payload, dict):
                row.update({key: value for key, value in payload.items() if not isinstance(value, (dict, list))})
            rows.append(row)
    if isinstance(data, dict) and isinstance(data.get("comparisons"), list):
        for payload in data["comparisons"]:
            row = {"artifact": path.name, "kind": kind, "section": "component_comparisons", "name": f"{payload.get('baseline')} -> {payload.get('candidate')}"}
            row.update({key: value for key, value in payload.items() if not isinstance(value, (dict, list))})
            rows.append(row)
    return pd.DataFrame(rows)


analysis_frames = []
for run_id in COMPARE_RUN_IDS:
    for path in json_analysis_artifacts(run_id):
        frame = summarize_analysis_artifact(path)
        if not frame.empty:
            frame.insert(0, "run_id", run_id)
            analysis_frames.append(frame)

analysis_summary_df = pd.concat(analysis_frames, ignore_index=True) if analysis_frames else pd.DataFrame()
if analysis_summary_df.empty:
    print("No JSON analysis summaries found for selected runs.")
else:
    display(analysis_summary_df)

## Failure Slices

Start with scorer-visible failures, then sort by clinical/debugging families. Keep examples row-level enough to explain behavior without turning the status file into a remediation log.

In [7]:
def failure_slices(frame: pd.DataFrame) -> dict[str, pd.DataFrame]:
    if frame.empty:
        return {}
    failures = frame[frame["purist_correct"] != True].copy()  # noqa: E712
    return {
        "by_gold_kind": (
            failures.groupby("gold_label_kind", dropna=False)
            .size()
            .rename("n")
            .reset_index()
        ),
        "by_prediction_kind": (
            failures.groupby("prediction_kind", dropna=False)
            .size()
            .rename("n")
            .reset_index()
        ),
        "by_confusion": (
            failures.groupby(["gold_purist_category", "predicted_purist_category"], dropna=False)
            .size()
            .rename("n")
            .reset_index()
            .sort_values("n", ascending=False)
        ),
        "parse_or_schema": frame[frame["parse_error_count"] > 0][
            ["source_row_index", "prediction_label", "gold_label", "parse_errors"]
        ],
        "evidence_invalid": frame[frame["evidence_valid"].eq(False)][
            ["source_row_index", "prediction_label", "gold_label", "selected_evidence"]
        ],
    }


slices = failure_slices(artifact_df)
for name, table in slices.items():
    print(f"\n{name}")
    display(table)


by_gold_kind


,gold_label_kind,n
0,frequency,8
1,unresolved_multiple,1



by_prediction_kind


,prediction_kind,n
0,frequency,9



by_confusion


,gold_purist_category,predicted_purist_category,n
2,seizure_freq_more1per6mon_less1mon,seizure_freq_more1mon_less1week,3
4,NaN,NaN,3
0,seizure_freq_more1mon_less1week,seizure_freq_1_per_mon,1
1,seizure_freq_more1mon_less1week,seizure_freq_more1week_less1day,1
3,seizure_freq_more1week_less1day,seizure_freq_more1mon_less1week,1



parse_or_schema


,source_row_index,prediction_label,gold_label,parse_errors
0,10,4 per day,4 per day,final_label_repaired: 'up to 4 per day' -> '4 per day'
1,40,4 per week,4 per week,final_label_repaired: '≤ 4 per week' -> '4 per week'
2,79,6 to 7 per year,6 to 7 per year,final_label_repaired: '≤ 6 to 7 per year' -> '6 to 7 per year'
5,156,1 per 6 day,1 per 6 day,final_label_repaired: '1 per 6 days' -> '1 per 6 day'
7,182,1 per 2 day,1 per 2 day,final_label_repaired: '1 per 2 days' -> '1 per 2 day'
8,187,1 cluster per week,1 per 7 to 9 day,unscorable_final_label: Unparsable cluster label: '1 cluster per week'
9,190,1 cluster per 4 week,1 per 4 week,final_label_repaired: '1 cluster per 4 weeks' -> '1 cluster per 4 week'; unscorable_final_label: Unparsable cluster label: '1 cluster pe...
12,218,1 per 3 week,1 per 3 week,final_label_repaired: '1 per 3 weeks' -> '1 per 3 week'
13,243,1 per 4 month,1 per 4 month,final_label_repaired: '1 per 4 months' -> '1 per 4 month'
17,409,1 per month,1 per month,final_label_repaired: '1 per month or less' -> '1 per month'



evidence_invalid


,source_row_index,prediction_label,gold_label,selected_evidence


## Row Review Queue

Use this compact queue for targeted row reading. Add columns as new artifact families expose richer diagnostics, but keep source text review disciplined by split policy.

In [8]:
if artifact_df.empty:
    review_queue = pd.DataFrame()
else:
    review_queue = (
        artifact_df[artifact_df["purist_correct"] != True]  # noqa: E712
        [[
            "source_row_index",
            "gold_label",
            "prediction_label",
            "gold_purist_category",
            "predicted_purist_category",
            "parse_errors",
            "selected_evidence",
            "rationale",
        ]]
        .sort_values(["parse_errors", "source_row_index"], ascending=[False, True])
        .reset_index(drop=True)
    )

review_queue

,source_row_index,gold_label,prediction_label,gold_purist_category,predicted_purist_category,parse_errors,selected_evidence,rationale
0,744,multiple per week,most weekdays,NaN,NaN,unscorable_final_label: Unparsable label (raw: 'most weekdays' / normalized: 'most weekdays'),"brief absences occurring on most weekdays, often clustering around late afternoon","The patient currently experiences frequent typical absence seizures on most weekdays, which represents the highest current seizure burde..."
1,187,1 per 7 to 9 day,1 cluster per week,NaN,NaN,unscorable_final_label: Unparsable cluster label: '1 cluster per week',events tend to cluster every seven to nine days,"The note indicates seizure events cluster every 7-9 days, which is the highest current seizure burden described. The overall cluster fre..."
2,190,1 per 4 week,1 cluster per 4 week,NaN,NaN,final_label_repaired: '1 cluster per 4 weeks' -> '1 cluster per 4 week'; unscorable_final_label: Unparsable cluster label: '1 cluster pe...,"he reports clusters of brief absence episodes every 4 weeks, usually over 1–2 days",Current seizure burden is best represented by the ongoing absence clusters every 4 weeks; GTCS are historical and currently absent.
3,212,1 per 3 to 4 week,1 per month,seizure_freq_more1mon_less1week,seizure_freq_1_per_mon,,ongoing episodes occurring every 3 - 4 weeks,"The note provides a clear current seizure frequency of ongoing episodes every 3-4 weeks, which is the highest current seizure burden des..."
4,665,2 per 2 week,2 per month,seizure_freq_more1week_less1day,seizure_freq_more1mon_less1week,,The app logs indicate a regular pattern of seizures twice every two weeks,"The note provides a clear, current, and quantified seizure frequency from the patient's seizure diary app over the past four months, whi..."
5,790,1 per 7 to 10 day,1 per week,seizure_freq_more1mon_less1week,seizure_freq_more1week_less1day,,"events have continued at a fairly regular cadence, occurring roughly once every seven to ten days","The overall seizure frequency is best represented by the patient's report of events occurring roughly once every seven to ten days, whic..."
6,959,1 per 2 month,2 per month,seizure_freq_more1per6mon_less1mon,seizure_freq_more1mon_less1week,,"She notes the events are occurring bimonthly on average, though some months she has none and then two in quick succession.","The note provides a clear current seizure frequency estimate as 'bimonthly on average' with corroborated diary data, indicating approxim..."
7,960,1 per 2 month,2 to 3 per month,seizure_freq_more1per6mon_less1mon,seizure_freq_more1mon_less1week,,ongoing events occurring with bimonthly seizures,"The note clearly states ongoing bimonthly seizures, indicating approximately 2 to 3 seizures per month, which is the highest current sei..."
8,987,1 per 2 month,2 per month,seizure_freq_more1per6mon_less1mon,seizure_freq_more1mon_less1week,,bimonthly seizures,"The note explicitly states 'bimonthly seizures' as the current overall seizure frequency despite adherence to medication, which is the c..."
